In [1]:
from test.helpers import SDE, simple_batch_sde_solve

import diffrax
import jax
import jax.numpy as jnp
import jax.random as jr
import numpy as np
from diffrax import (
    ControlTerm,
    MultiTerm,
    ODETerm,
    ShARK,
    SpaceTimeLevyArea,
    VirtualBrownianTree,
)
from jax import Array


jax.config.update("jax_enable_x64", True)

vbt = VirtualBrownianTree(
    0.0,
    16.0,
    0.001,
    (),
    jr.key(8),
    SpaceTimeLevyArea,
    _w_t1=jnp.array(-3.0),
    _hh_t1=jnp.array(5.0),
)
out = vbt.evaluate(0.0, 3.0, use_levy=True)
print(out)
print(out.W)
print(out.H)

SpaceTimeLevyArea(dt=f64[], W=f64[], H=f64[])
4.127056709441923
-0.35470738388700485


In [2]:
# sst_sde is two-dimensional, the first dimension being W_t, the second \int_0^t W_s ds


def drift(t, y, args):
    return jnp.array([0.0, y[0] ** 2], dtype=y.dtype)


def diffusion(t, y, args):
    return jnp.array([1.0, 0.0], dtype=y.dtype)


def get_terms(bm):
    return MultiTerm(ODETerm(drift), ControlTerm(diffusion, bm))


y0 = jnp.array([0.0, 0.0], dtype=jnp.float64)
args = None
t0 = 0.0
t1 = 1.0
bm_shape = ()

sst_sde = SDE(get_terms, args, y0, t0, t1, bm_shape)

In [3]:
def compute_sst(w, hh, num_samples, key):
    keys = jr.split(key, num_samples)
    saveat = diffrax.SaveAt(t1=True)
    bm_tol = 2**-6
    dt0 = 2**-4
    solver = ShARK()
    controller = None
    levy_area = SpaceTimeLevyArea
    w = jnp.array(w, dtype=jnp.float64)
    hh = jnp.array(hh, dtype=jnp.float64)
    y1, _ = simple_batch_sde_solve(
        keys,
        sst_sde,
        solver,
        levy_area,
        dt0,
        controller,
        bm_tol,
        saveat,
        _w_t1=w,
        _hh_t1=hh,
    )
    sst = jnp.squeeze(y1)[:, 1]
    return sst


def true_cond_stats_c(w, hh):
    w2 = w**2
    hh2 = hh**2
    mean = 1 / 3 * w2 + w * hh + 6 / 5 * hh2 + 1 / 15
    var = 11 / 6300 + 1 / 180 * w2 + 1 / 175 * hh2
    return mean, var


def stat_error(c: Array, w, hh):
    w = float(w)
    hh = float(hh)
    true_mean, true_var = true_cond_stats_c(w, hh)
    mean_error = jnp.abs(true_mean - jnp.mean(c))
    var_error = jnp.abs(true_var - jnp.var(c))
    return mean_error, var_error

In [4]:
sst = compute_sst(3.0, 5.0, 2**5, jr.PRNGKey(0))
mean_error, var_error = stat_error(sst, 3.0, 5.0)
print(f"Mean error: {mean_error}, variance error: {var_error}")

Mean error: 0.13179540768994258, variance error: 0.017295123843921095


In [6]:
# Save w, hh and sst to a file
def save_sst(w, hh, sst: Array):
    w_flt = float(w)
    hh_flt = float(hh)
    w = np.array(w)
    hh = np.array(hh)
    sst = np.array(sst)
    file_name = f"sst_saved_values/sst_w{w_flt:.3}_hh{hh_flt:.3}.npy"
    with open(file_name, "wb") as f:
        np.save(f, w)
        np.save(f, hh)
        np.save(f, sst)


def load_sst(w_flt: float, hh_flt: float):
    file_name = f"sst_saved_values/sst_w{w_flt:.3}_hh{hh_flt:.3}.npy"
    with open(file_name, "rb") as f:
        w = np.load(f)
        hh = np.load(f)
        sst = np.load(f)

    return w, hh, sst


save_sst(3.0, 5.0, sst)
w_loaded, hh_loaded, sst_loaded = load_sst(3.0, 5.0)
print(w_loaded)
print(hh_loaded)
print(sst_loaded)

3.0
5.0
[48.4489412  48.7243649  48.80994662 48.50987155 48.72254373 47.87837759
 48.10262205 48.03555665 47.96121341 48.29420197 47.8128399  48.67641868
 48.42671913 48.26970205 48.47301597 48.21505293 47.80002186 48.60445313
 48.31342527 47.9222807  47.95787968 49.19755831 47.29807192 48.12429317
 48.86317405 47.62290123 47.86300233 48.31843235 47.23655591 47.38728034
 47.9158848  48.56418299]
